In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 87.9 MB/s eta 0:00:00


In [ ]:
import kagglehub

path = kagglehub.dataset_download("karkavelrajaj/amazon-sales-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'amazon-sales-dataset' dataset.
Path to dataset files: /kaggle/input/amazon-sales-dataset


In [ ]:
import pandas as pd
import numpy as np
import os
csv_path = os.path.join(path, "amazon.csv")

df = pd.read_csv(csv_path)

print(df.head())

   product_id                                       product_name  \
0  B07JW9H4J1  Wayona Nylon Braided USB to Lightning Fast Cha...   
1  B098NS6PVG  Ambrane Unbreakable 60W / 3A Fast Charging 1.5...   
2  B096MSW6CT  Sounce Fast Phone Charging Cable & Data Sync U...   
3  B08HDJ86NZ  boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...   
4  B08CF3B7N1  Portronics Konnect L 1.2M Fast Charging 3A 8 P...   

                                            category discounted_price  \
0  Computers&Accessories|Accessories&Peripherals|...             ₹399   
1  Computers&Accessories|Accessories&Peripherals|...             ₹199   
2  Computers&Accessories|Accessories&Peripherals|...             ₹199   
3  Computers&Accessories|Accessories&Peripherals|...             ₹329   
4  Computers&Accessories|Accessories&Peripherals|...             ₹154   

  actual_price discount_percentage rating rating_count  \
0       ₹1,099                 64%    4.2       24,269   
1         ₹349                 43%  

In [ ]:
data = []
for _, row in df.iterrows():
  doc = f"""
    product: {row['product_name']}
    Category: {row['category']}
    Discounted Price: {row['discounted_price']}
    Rating: {row['rating']}
    Rating Count: {row['rating_count']}
    About: {row['about_product']}
    Reviews: {row['review_content']}
  """
  data.append(doc)

In [ ]:
from sentence_transformers import SentenceTransformer
embed = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embedding = embed.encode(data)

In [ ]:
dimension = embedding.shape[1]
dimension

384

In [ ]:
import faiss
import numpy as np
faiss.normalize_L2(embedding)
index = faiss.IndexFlatIP(dimension)

In [ ]:
index.add(embedding)

In [ ]:
query = "give me Good Bluetooth Speaker"
query_embedding = embed.encode([query])

In [ ]:
query_embedding.dtype

dtype('float32')

In [ ]:
print(query_embedding.shape)

(1, 384)


In [ ]:
faiss.normalize_L2(query_embedding)
distance, indices = index.search(query_embedding, 5)

In [ ]:
print(distance, indices)

[[0.6206144  0.6123325  0.58254063 0.5823822  0.5759572 ]] [[753 712 730 716 999]]


In [ ]:
reterived_chunk = []
for i in indices[0]:
  reterived_chunk.append(data[i])

In [ ]:
rag_memory = "\n\n".join(reterived_chunk)
rag_memory

"\n    product: Infinity (JBL Fuze Pint, Wireless Ultra Portable Mini Speaker with Mic, Deep Bass, Dual Equalizer, Bluetooth 5.0 with Voice Assistant Support for Mobiles (Black)\n    Category: Electronics|HomeAudio|Speakers|OutdoorSpeakers\n    Discounted Price: ₹899\n    Rating: 4.1\n    Rating Count: 30,469\n    About: Pocket Size Portable Bluetooth Speakers|5 hours Music Playtime Under Optimum Audio Settings and please ensure speaker is 100% charged before usage|Dual Equalizer Modes for Normal & Deep Bass Output|Wireless Bluetooth Streaming|Speakerphone. Frequency Response 180Hz - 20KHz. Signal to noise Ratio 70dB (Aux)|Voice Assistant Integration|Battery Size (mAh) 3.7V/480mAH with Charging Time 2.5 H @ 5V0.5A\n    Reviews: The speaker's sound is good and all the other feature are also at the best. the only problem is the battery backup it's normally about 5hr, but that can be acceptable with this price.,Small but over all good,Price is too high for this product but sound quality i

In [ ]:
GEMINI_API_KEY = ""

In [ ]:
import google.generativeai as genai
genai.configure(api_key = GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")
text = f"""
  Your a Expert in Shopping you have to suggest used based on the user query and reterive items
  user query:
  {query}
  Reterived Products:
  {rag_memory}
  Instruction:
  1. Analyse the user Requirements
  2. suggest the best products
  3. mention the price, link, rating and Important features
  Note:
  Give the output in the structural Format
  Dont use * and #
"""
responce = model.generate_content(text)
print(responce.text)

As an Expert in Shopping, I understand you're looking for a "Good Bluetooth Speaker." To give you the best recommendations, I've analyzed the provided products based on sound quality, portability, battery life, features, and overall value.

Here are my top suggestions for Good Bluetooth Speakers:

Product Recommendation 1: JBL Go 2, Wireless Portable Bluetooth Speaker

Price: ₹1,999
Rating: 4.3 out of 5 (from 63,899 ratings)

Important Features:
  *   JBL Signature Sound: Known for clear and balanced audio.
  *   IPX7 Waterproof: Can be submerged in water, making it ideal for pool parties, beach trips, or shower use.
  *   5 Hours of Playtime: Decent battery life for its compact size.
  *   Built-in Noise-cancelling Speakerphone: For clear hands-free calls.
  *   AUX Input: Allows wired connection to devices without Bluetooth.
  *   Very Portable: Small and easy to carry.

Why I recommend it: The JBL Go 2 stands out for its superior sound quality for its size, excellent brand reliabili

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = ""

In [ ]:
from google.adk.flows.llm_flows import instructions
from google.adk.agents import Agent
from google.adk.tools import google_search
root_agent = Agent(
    name = "google_search_agent",
    model = "gemini-2.5-flash",
    instruction = "Answer User Question using Google Search",
    tools = [google_search]
)

/usr/local/lib/python3.12/dist-packages/IPython/lib/py311_tokenize.py:532: RuntimeWarning: coroutine 'search_web' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)


In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import asyncio

APP_NAME = "Web_search"
USER_ID = "kirit"
SESSION_ID = "session3"

session_service = InMemorySessionService()
runner = Runner(
    agent = root_agent,
    app_name = APP_NAME,
    session_service = session_service
)

In [ ]:
async def search_web(query):
  await session_service.create_session(
      app_name = APP_NAME,
      user_id = USER_ID,
      session_id = SESSION_ID,
  )
  content = types.Content(
      role = "user",
      parts = [types.Part(text = query)]
  )
  async for event in runner.run_async(
      user_id = USER_ID,
      session_id = SESSION_ID,
      new_message = content
  ):
    if event.is_final_response():
      return event.content.parts[0].text

In [ ]:
query = "give me Good Bluetooth Speaker"

In [ ]:
responce_agent = await search_web(query)

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-63' coro=<BaseApiClient.aclose() done, defined at /usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py:1963> exception=AttributeError("'BaseApiClient' object has no attribute '_async_httpx_client'")>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1968, in aclose
    await self._async_httpx_client.aclose()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'BaseApiClient' object has no attribute '_async_httpx_client'


In [ ]:
import google.generativeai as genai
genai.configure(api_key = GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")
text = f"""
  Your a Expert in Shopping you have to suggest used based on the user query and reterive items
  user query:
  {query}
  Reterived Products:
  {responce_agent}
  Instruction:
  1. Analyse the user Requirements
  2. suggest the best products
  3. mention the price, link, rating and Important features
  Note:
  Give the output in the structural Format
  Dont use * and #
"""
responce = model.generate_content(text)
print(responce.text)

Hello! As an expert in shopping, I understand you're looking for a good Bluetooth speaker. The term "good" can mean different things to different people, depending on priorities like sound quality, portability, durability, battery life, and budget. Based on your general query, I've selected a few highly-regarded options that excel in different aspects, offering a well-rounded selection to help you find your perfect match.

Here are my top recommendations:

---

**Recommendation 1: JBL Flip 7**

The JBL Flip 7 is an excellent all-rounder, often hailed as one of the best overall choices for its impressive balance of features. It's a fantastic option for most users looking for reliable performance.

Price: Approximately $100 - $130 (Prices can vary, check current retail offers)
Rating: 4.6/5 (Based on typical customer reviews across major retailers)
Link: Available at major electronics retailers such as Amazon, Best Buy, and Walmart.

Important Features:
*   Great balance of portability, 